In [ ]:
import matplotlib.pyplot as plt
import anndata as adata
import scanpy as sc
from wmb import cemba, mm10
from ALLCools.mcds import MCDS
from ALLCools.clustering import significant_pc_test
from ALLCools.plot import *
import pandas as pd
from harmonypy import run_harmony
import loompy as lp
import numpy as np
import glob
import seaborn as sns
# load the regulons from a file using the load_signatures function
from pyscenic.cli.utils import load_signatures
from pyscenic.export import add_scenic_metadata
from pyscenic.utils import modules_from_adjacencies
import json
import zlib
import base64
from pyscenic.utils import load_motifs
import operator as op
from IPython.display import HTML, display
from pyscenic.rss import regulon_specificity_scores
from pyscenic.plotting import plot_rss
import matplotlib.pyplot as plt
from adjustText import adjust_text
import seaborn as sns
from pyscenic.binarization import binarize
import matplotlib as mpl



In [ ]:
adata_sc = adata.read_h5ad("/data1st1/hydata/pfc_3tech_int_byID.h5ad")

In [ ]:
adata_sc_selected = adata_sc[(adata_sc.obs.tech == '10x') & (adata_sc.obs.group == 'W')]

In [ ]:
df_annotation = pd.read_csv('output/sc-celltype-annotated.csv', index_col=0)

In [ ]:
df_annotation.index =df_annotation.index.str.split('-').str[:-1].str.join('-')

In [ ]:
adata_sc_selected.obs = df_annotation.loc[adata_sc_selected.obs.index]

In [ ]:
adata_sc_selected.layers['counts']

In [ ]:
df_gene_names = pd.read_csv('output/sc_gene_ids_meta.csv',index_col=0)

In [ ]:
adata_sc_selected.var = df_gene_names.loc[adata_sc_selected.var.index]

In [ ]:
adata_sc_selected.var_names

In [ ]:
counts = adata_sc_selected.layers['counts'].copy()

In [ ]:
adata_sc_selected.var

In [ ]:
row_attrs = {
    "Gene": np.array(adata_sc_selected.var_names),
    "GeneID": np.array(adata_sc_selected.var.gene_ids),
}
col_attrs = {
    "CellID": np.array(adata_sc_selected.obs_names) ,
    "nGene": np.array( np.sum(counts.transpose()>0 , axis=0)).flatten() ,
    "nUMI": np.array( np.sum(counts.transpose() , axis=0)).flatten() ,
}


In [ ]:
f_loom_path_scenic = "output/pfc_10x_scenic.loom"
lp.create( f_loom_path_scenic, counts.transpose(), row_attrs, col_attrs)


In [ ]:
f_tfs="/home/junyichen/code/scmmd/data/allTFs_mm.txt"

In [ ]:
!pyscenic grn {f_loom_path_scenic} {f_tfs} -o adj.csv --num_workers 20


In [ ]:
adjacencies = pd.read_csv("output/adj.csv", index_col=False)

In [ ]:
adjacencies

In [ ]:
f_db_glob = "/data2st1/junyi/scenic/mouse/motif/*feather"
f_db_names = ' '.join( glob.glob(f_db_glob) )
f_motif_path = "/data2st1/junyi/scenic/mouse/motif/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl"


In [ ]:
!pyscenic ctx "output/adj.csv" \
    {f_db_names} \
    --annotations_fname {f_motif_path} \
    --expression_mtx_fname {f_loom_path_scenic} \
    --output "output/reg.csv" \
    --mask_dropouts \
    --num_workers 20

In [ ]:
nGenesDetectedPerCell = np.asarray(np.sum(counts>0, axis=1).flatten())[0]
percentiles = np.quantile(nGenesDetectedPerCell, [.01, .05, .10, .50, .99])
#percentiles = nGenesDetectedPerCell.quantile([.01, .05, .10, .50, 1])

percentiles_index_values =[.01, .05, .10, .50, .99]
print(percentiles)


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5), dpi=150)
sns.distplot(nGenesDetectedPerCell, norm_hist=False, kde=False, bins='fd')
ax.set_xlim(0, 5000)
for i,x in enumerate(percentiles):
    fig.gca().axvline(x=x, ymin=0,ymax=1, color='red')
    ax.text(x=x, y=ax.get_ylim()[1], s=f'{int(x)} ({percentiles_index_values[i]*100}%)', color='red', rotation=30, size='x-small',rotation_mode='anchor' )
ax.set_xlabel('# of genes')
ax.set_ylabel('# of cells')
fig.tight_layout()


In [ ]:
f_pyscenic_output = "output/pyscenic_output.loom"


In [ ]:
!pyscenic aucell \
    {f_loom_path_scenic} \
    output/reg.csv \
    --output {f_pyscenic_output} \
    --num_workers 20


In [ ]:


# collect SCENIC AUCell output
lf = lp.connect( f_pyscenic_output, mode='r+', validate=False )
auc_mtx = pd.DataFrame( lf.ca.RegulonsAUC, index=lf.ca.CellID)
lf.close()


In [ ]:
import umap

# UMAP
runUmap = umap.UMAP(n_neighbors=10, min_dist=0.4, metric='correlation').fit_transform
dr_umap = runUmap( auc_mtx )
pd.DataFrame(dr_umap, columns=['X', 'Y'], index=auc_mtx.index).to_csv( "output/scenic_umap.txt", sep='\t')
# tSNE
# tsne = TSNE( n_jobs=20 )
# dr_tsne = tsne.fit_transform( auc_mtx )
# pd.DataFrame(dr_tsne, columns=['X', 'Y'], index=auc_mtx.index).to_csv( "scenic_tsne.txt", sep='\t')


In [ ]:
lf = lp.connect( f_pyscenic_output, mode='r+', validate=False )
meta = json.loads(zlib.decompress(base64.b64decode( lf.attrs.MetaData )))
#exprMat = pd.DataFrame( lf[:,:], index=lf.ra.Gene, columns=lf.ca.CellID)
auc_mtx = pd.DataFrame( lf.ca.RegulonsAUC, index=lf.ca.CellID)
regulons = lf.ra.Regulons
dr_umap = pd.read_csv( 'output/scenic_umap.txt', sep='\t', header=0, index_col=0 )
# dr_tsne = pd.read_csv( 'scenic_tsne.txt', sep='\t', header=0, index_col=0 )
###


In [ ]:
auc_mtx.columns = auc_mtx.columns.str.replace('\(','_(')
regulons.dtype.names = tuple( [ x.replace("(","_(") for x in regulons.dtype.names ] )
# regulon thresholds
rt = meta['regulonThresholds']
for i,x in enumerate(rt):
    tmp = x.get('regulon').replace("(","_(")
    x.update( {'regulon': tmp} )


In [ ]:
# tsneDF = pd.DataFrame(adata.obsm['X_tsne'], columns=['_X', '_Y'])

Embeddings_X = pd.DataFrame( index=lf.ca.CellID )
Embeddings_X = pd.concat( [
        pd.DataFrame(adata_sc_selected.obsm['X_umap'],index=adata_sc_selected.obs.index)[0] ,
        pd.DataFrame(adata_sc_selected.obsm['X_pca'],index=adata_sc_selected.obs.index)[0] ,
        # dr_tsne['X'] ,
        dr_umap['X']
    ], sort=False, axis=1, join='outer' )
Embeddings_X.columns = ['1','2','3']

Embeddings_Y = pd.DataFrame( index=lf.ca.CellID )
Embeddings_Y = pd.concat( [
        pd.DataFrame(adata_sc_selected.obsm['X_umap'],index=adata_sc_selected.obs.index)[1] ,
        pd.DataFrame(adata_sc_selected.obsm['X_pca'],index=adata_sc_selected.obs.index)[1] ,
        # dr_tsne['Y'] ,
        dr_umap['Y']
    ], sort=False, axis=1, join='outer' )
Embeddings_Y.columns = ['1','2','3']


In [ ]:
adata_sc_selected.obs['leiden'].values.astype(str)

In [ ]:
metaJson = {}

metaJson['embeddings'] = [
    # {
    #     "id": -1,
    #     "name": f"Scanpy t-SNE (highly variable genes)"
    # },
    {
        "id": 1,
        "name": f"Scanpy UMAP  (highly variable genes)"
    },
    {
        "id": 2,
        "name": "Scanpy PC1/PC2"
    },
    # {
    #     "id": 3,
    #     "name": "SCENIC AUC t-SNE"
    # },
    {
        "id": 3,
        "name": "SCENIC AUC UMAP"
    },
]

metaJson["clusterings"] = [{
            "id": 0,
            "group": "Scanpy",
            "name": "Scanpy leiden default resolution",
            "clusters": [],
        }]

metaJson["metrics"] = [
        {
            "name": "nUMI"
        }, {
            "name": "nGene"
        }, {
            "name": "Percent_mito"
        }
]

metaJson["annotations"] = [
    {
        "name": "Leiden_clusters_Scanpy",
        "values": list(set( adata_sc_selected.obs['leiden'].values.astype(str)))

    },
    {
       "name": "CellTypeL1",
       "values": list(set(adata_sc_selected.obs['celltype.L1'].values))
    },
    {
       "name": "CellTypeL2",
       "values": list(set(adata_sc_selected.obs['celltype.L2'].values))
    },
    #{
    #    "name": "Sample",
    #    "values": list(set(adata.obs['Sample'].values))
    #}
]

# SCENIC regulon thresholds:
metaJson["regulonThresholds"] = rt

for i in range(max(set([int(x) for x in adata_sc_selected.obs['leiden']])) + 1):
    clustDict = {}
    clustDict['id'] = i
    clustDict['description'] = f'Unannotated Cluster {i + 1}'
    metaJson['clusterings'][0]['clusters'].append(clustDict)
    
clusterings = pd.DataFrame()
clusterings["0"] = adata_sc_selected.obs['louvain'].values.astype(np.int64)


In [ ]:
def dfToNamedMatrix(df):
    arr_ip = [tuple(i) for i in df.values]
    dtyp = np.dtype(list(zip(df.dtypes.index, df.dtypes)))
    arr = np.array(arr_ip, dtype=dtyp)
    return arr


In [ ]:
umapDF = pd.DataFrame(adata_sc_selected.obsm['X_umap'], columns=['_X', '_Y'])


In [ ]:
col_attrs = {
    "CellID": np.array(adata_sc_selected.obs.index),
    "nUMI": np.array(adata_sc_selected.obs['total_counts'].values),
    "nGene": np.array(adata_sc_selected.obs['n_genes_by_counts'].values),
    "Leiden_clusters_Scanpy": np.array( adata_sc_selected.obs['leiden'].values ),
    "CellTypeL1": np.array(adata_sc_selected.obs['celltype.L1'].values),
    "CellTypeL2": np.array(adata_sc_selected.obs['celltype.L2'].values),
    #"Sample": np.array(adata.obs['Sample'].values),
    "Percent_mito": np.array(adata_sc_selected.obs['pct_counts_mt'].values),
    "Embedding": dfToNamedMatrix(umapDF),
    "Embeddings_X": dfToNamedMatrix(Embeddings_X),
    "Embeddings_Y": dfToNamedMatrix(Embeddings_Y),
    "RegulonsAUC": dfToNamedMatrix(auc_mtx),
    "Clusterings": dfToNamedMatrix(clusterings),
    "ClusterID": np.array(adata_sc_selected.obs['leiden'].values)
}

row_attrs = {
    "Gene": lf.ra.Gene,
    "Regulons": regulons,
}

attrs = {
    "title": "sampleTitle",
    "MetaData": json.dumps(metaJson),
    "Genome": 'mm10',
    "SCopeTreeL1": "",
    "SCopeTreeL2": "",
    "SCopeTreeL3": ""
}

# compress the metadata field:
attrs['MetaData'] = base64.b64encode(zlib.compress(json.dumps(metaJson).encode('ascii'))).decode('ascii')


In [ ]:
f_final_loom = 'output/pfc_scenic_integrated-output.loom'

In [ ]:
lp.create(
    filename = f_final_loom ,
    layers=lf[:,:],
    row_attrs=row_attrs, 
    col_attrs=col_attrs, 
    file_attrs=attrs
)
lf.close() # close original pyscenic loom file


# Post processing of the loom file

In [ ]:
f_final_loom = 'output/pfc_scenic_integrated-output.loom'
sc.settings.verbosity = 3 # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_versions()
sc.settings.set_figure_params(dpi=150)


In [ ]:
lf = lp.connect( f_final_loom, mode='r', validate=False )
meta = json.loads(zlib.decompress(base64.b64decode( lf.attrs.MetaData )))
exprMat = pd.DataFrame( lf[:,:], index=lf.ra.Gene, columns=lf.ca.CellID).T
auc_mtx = pd.DataFrame( lf.ca.RegulonsAUC, index=lf.ca.CellID)


In [ ]:
regulons = {}
for i,r in pd.DataFrame(lf.ra.Regulons,index=lf.ra.Gene).iteritems():
    regulons[i] =  list(r[r==1].index.values)


In [ ]:
lf.ca.keys()

In [ ]:
cellAnnot = pd.concat(
    [
        pd.DataFrame( lf.ca.CellTypeL1, index=lf.ca.CellID ),
        pd.DataFrame( lf.ca.ClusterID, index=lf.ca.CellID ),
        pd.DataFrame( lf.ca.Leiden_clusters_Scanpy, index=lf.ca.CellID ),
        pd.DataFrame( lf.ca.Percent_mito, index=lf.ca.CellID ),
        pd.DataFrame( lf.ca.nGene, index=lf.ca.CellID ),
        pd.DataFrame( lf.ca.nUMI, index=lf.ca.CellID ),
    ],
    axis=1
)
cellAnnot.columns = [
 'CellTypeL1',
 'ClusterID',
 'Leiden_clusters_Scanpy',
 'Percent_mito',
 'nGene',
 'nUMI']


In [ ]:
# capture embeddings:
dr = [
    pd.DataFrame( lf.ca.Embedding, index=lf.ca.CellID )
]
dr_names = [
    meta['embeddings'][0]['name'].replace(" ","_")
]

# add other embeddings
drx = pd.DataFrame( lf.ca.Embeddings_X, index=lf.ca.CellID )
dry = pd.DataFrame( lf.ca.Embeddings_Y, index=lf.ca.CellID )

for i in range( len(drx.columns) ):
    dr.append( pd.concat( [ drx.iloc[:,i], dry.iloc[:,i] ], sort=False, axis=1, join='outer' ))
    dr_names.append( meta['embeddings'][i]['name'].replace(" ","_").replace('/','-') )

# rename columns:
for i,x in enumerate( dr ):
    x.columns = ['X','Y']


In [ ]:
lf.close()


In [ ]:
adata = sc.read( f_final_loom, validate=False)


In [ ]:
for i,x in enumerate( dr ):
    adata.obsm[ 'X_'+dr_names[i] ] = x.values


In [ ]:
categorical_data = adata.obs.select_dtypes(include=['category'])

# Fetch numerical data
numerical_data = adata.obs.select_dtypes(include=['number'])

# Fetch string (object) data
string_data = adata.obs.select_dtypes(include=['object'])


In [ ]:
adata.obs=categorical_data.merge(numerical_data, left_index=True, right_index=True).merge(string_data, left_index=True, right_index=True)

In [ ]:
sc._utils.sanitize_anndata( adata )


In [ ]:


sig = load_signatures('output/reg.csv')
adata = add_scenic_metadata(adata, auc_mtx, sig)


In [ ]:


BASE_URL = "http://motifcollections.aertslab.org/v9/logos/"
COLUMN_NAME_LOGO = "MotifLogo"
COLUMN_NAME_MOTIF_ID = "MotifID"
COLUMN_NAME_TARGETS = "TargetGenes"

def display_logos(df: pd.DataFrame, top_target_genes: int = 3, base_url: str = BASE_URL):
    """
    :param df:
    :param base_url:
    """
    # Make sure the original dataframe is not altered.
    df = df.copy()
    
    # Add column with URLs to sequence logo.
    def create_url(motif_id):
        return '<img src="{}{}.png" style="max-height:124px;"></img>'.format(base_url, motif_id)
    df[("Enrichment", COLUMN_NAME_LOGO)] = list(map(create_url, df.index.get_level_values(COLUMN_NAME_MOTIF_ID)))
    
    # Truncate TargetGenes.
    def truncate(col_val):
        return sorted(col_val, key=op.itemgetter(1))[:top_target_genes]
    df[("Enrichment", COLUMN_NAME_TARGETS)] = list(map(truncate, df[("Enrichment", COLUMN_NAME_TARGETS)]))
    
    MAX_COL_WIDTH = pd.get_option('display.max_colwidth')
    pd.set_option('display.max_colwidth', 200)
    display(HTML(df.head().to_html(escape=False)))
    pd.set_option('display.max_colwidth', MAX_COL_WIDTH)


In [ ]:
df_motifs = load_motifs('output/reg.csv')


In [ ]:
adata

In [ ]:
sc.set_figure_params(frameon=False, dpi=600, fontsize=10, dpi_save=600)

sc.pl.scatter( adata, basis='Scanpy_UMAP__(highly_variable_genes)', 
    color=['CellTypeL1','CellTypeL2'],
    title=['HVG - UMAP (Louvain clusters)','HVG - UMAP (Cell type)'],
    alpha=0.8,
    save='_Louvain-celltype.pdf'
    )

sc.pl.scatter( adata, basis='SCENIC_AUC_UMAP', 
    color=['CellTypeL1','CellTypeL2'],
    title=['SCENIC - UMAP (Louvain clusters)','SCENIC - UMAP (Cell type)'], 
    alpha=0.8,
    save='_Louvain-celltype.pdf'
    )

In [ ]:
rss_cellType = regulon_specificity_scores( auc_mtx, cellAnnot['CellTypeL1'] )
rss_cellType

In [ ]:
cats = sorted(list(set(cellAnnot['CellTypeL1'])))

fig = plt.figure(figsize=(15, 8))
for c,num in zip(cats, range(1,len(cats)+1)):
    x=rss_cellType.T[c]
    ax = fig.add_subplot(2,5,num)
    plot_rss(rss_cellType, c, top_n=5, max_n=None, ax=ax)
    ax.set_ylim( x.min()-(x.max()-x.min())*0.05 , x.max()+(x.max()-x.min())*0.05 )
    for t in ax.texts:
        t.set_fontsize(12)
    ax.set_ylabel('')
    ax.set_xlabel('')
    adjust_text(ax.texts, autoalign='xy', ha='right', va='bottom', arrowprops=dict(arrowstyle='-',color='lightgrey'), precision=0.001 )
 
fig.text(0.5, 0.0, 'Regulon', ha='center', va='center', size='x-large')
fig.text(0.00, 0.5, 'Regulon specificity score (RSS)', ha='center', va='center', rotation='vertical', size='x-large')
plt.tight_layout()
plt.rcParams.update({
    'figure.autolayout': True,
        'figure.titlesize': 'large' ,
        'axes.labelsize': 'medium',
        'axes.titlesize':'large',
        'xtick.labelsize':'medium',
        'ytick.labelsize':'medium'
        })
plt.savefig("figures/PFC_cellType-RSS-top5.pdf", dpi=600, bbox_inches = "tight")
plt.show()

In [ ]:
topreg = []
for i,c in enumerate(cats):
    topreg.extend(
        list(rss_cellType.T[c].sort_values(ascending=False)[:5].index)
    )
topreg = list(set(topreg))

In [ ]:
auc_mtx_Z = pd.DataFrame( index=auc_mtx.index )
for col in list(auc_mtx.columns):
    auc_mtx_Z[ col ] = ( auc_mtx[col] - auc_mtx[col].mean()) / auc_mtx[col].std(ddof=0)
#auc_mtx_Z.sort_index(inplace=True)

In [ ]:
def palplot(pal, names, colors=None, size=1):
    n = len(pal)
    f, ax = plt.subplots(1, 1, figsize=(n * size, size))
    ax.imshow(np.arange(n).reshape(1, n),
              cmap=mpl.colors.ListedColormap(list(pal)),
              interpolation="nearest", aspect="auto")
    ax.set_xticks(np.arange(n) - .5)
    ax.set_yticks([-.5, .5])
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    colors = n * ['k'] if colors is None else colors
    for idx, (name, color) in enumerate(zip(names, colors)):
        ax.text(0.0+idx, 0.0, name, color=color, horizontalalignment='center', verticalalignment='center')
    return f

In [ ]:
colors = sns.color_palette('bright',n_colors=len(cats) )
colorsd = dict( zip( cats, colors ))
colormap = [ colorsd[x] for x in cellAnnot['CellTypeL1'] ]

In [ ]:
sns.set()
sns.set(font_scale=0.8)
fig = palplot( colors, cats, size=1.0)
plt.savefig("figures/PFC_cellType-heatmap-legend-top5.pdf", dpi=600, bbox_inches = "tight")

In [ ]:
sns.set(font_scale=1.2)
g = sns.clustermap(auc_mtx_Z[topreg], annot=False,  square=False,  linecolor='gray',
    yticklabels=False, xticklabels=True, vmin=-2, vmax=6, row_colors=colormap,
    cmap="YlGnBu", figsize=(21,16) )
g.cax.set_visible(True)
g.ax_heatmap.set_ylabel('')
g.ax_heatmap.set_xlabel('')
plt.savefig("figures/PFC_cellType-heatmap-top5.pdf", dpi=600, bbox_inches = "tight")

In [ ]:
binary_mtx, auc_thresholds = binarize( auc_mtx, num_workers=25 )
binary_mtx.head()

In [ ]:
top_RSS={}
for col in rss_cellType.index:
    top_RSS[col] = rss_cellType.T[col].sort_values(ascending=False)[:5].index.values

In [ ]:
top_RSS

In [ ]:
r = topreg[::5]

fig, axs = plt.subplots(1, 5, figsize=(12, 4), dpi=150, sharey=False)
for i,ax in enumerate(axs):
    sns.distplot(auc_mtx[ r[i] ], ax=ax, norm_hist=True, bins=100)
    ax.plot( [ auc_thresholds[ r[i] ] ]*2, ax.get_ylim(), 'r:')
    ax.title.set_text( r[i] )
    ax.set_xlabel('')
    
fig.text(-0.01, 0.5, 'Frequency', ha='center', va='center', rotation='vertical', size='large')
fig.text(0.5, -0.01, 'AUC', ha='center', va='center', rotation='horizontal', size='large')

fig.tight_layout()
fig.savefig('figures/PFC_cellType-binaryPlot2.pdf', dpi=600, bbox_inches='tight')

In [ ]:
adjacencies = pd.read_csv("output/adj.csv", index_col=False)

In [ ]:
modules = list(modules_from_adjacencies(adjacencies, exprMat))

In [ ]:
topreg

In [ ]:
for tf in topreg:
    tf = tf.split("_")[0]
    tf_mods = [ x for x in modules if x.transcription_factor==tf ]

    for i,mod in enumerate( tf_mods ):
        print( f'{tf} module {str(i)}: {len(mod.genes)} genes' )
    print( f'{tf} regulon: {len(regulons[tf+"_(+)"])} genes' )
    for i,mod in enumerate( tf_mods ):
        with open( "output/"+tf+'_module_'+str(i)+'.txt', 'w') as f:
            for item in mod.genes:
                f.write("%s\n" % item)
                
    with open( "output/"+tf+'_regulon.txt', 'w') as f:
        for item in regulons[tf+'_(+)']:
            f.write("%s\n" % item)

In [ ]:
import networkx as nx


In [ ]:
regulons

In [ ]:
regulons.keys()

In [ ]:
top_RSS.items()

In [ ]:
cell_type_graphs = {}


for cell_type, top_regulons in top_RSS.items():
    G = nx.DiGraph()
    for regulon in top_regulons:
        G.add_node(regulon, type="TF", cell_type=cell_type)
        target_genes = regulons.get(regulon, [])
        G.add_nodes_from(target_genes, type="target", cell_type=cell_type)
        G.add_edges_from([(regulon, gene) for gene in target_genes])
    cell_type_graphs[cell_type] = G

In [ ]:
merged_graph = nx.compose_all(cell_type_graphs.values())


In [ ]:
import matplotlib.pyplot as plt

# Draw the merged graph
plt.figure(figsize=(12, 12))
pos = nx.spring_layout(merged_graph, k=0.15)  # Layout for positioning nodes

# Color nodes by type (TF or target)
node_colors = ["lightblue" if merged_graph.nodes[node]["type"] == "TF" else "lightgreen" for node in merged_graph.nodes]

# Draw nodes and edges
nx.draw_networkx_nodes(merged_graph, pos, node_size=300, node_color=node_colors, alpha=0.8)
nx.draw_networkx_edges(merged_graph, pos, edge_color="gray", alpha=0.4, arrowstyle="->", arrowsize=10)
nx.draw_networkx_labels(merged_graph, pos, font_size=8)

plt.title("Merged Regulon Network for All Cell Types")
plt.axis("off")
plt.show()

In [ ]:
regulon_name = list(regulons.keys())[0]  # Get the first TF name
target_genes = regulons[regulon_name]    # Get the target genes for this TF


In [ ]:
G = nx.DiGraph()
G.add_node(regulon_name, type="TF")
G.add_nodes_from(target_genes, type="target")
G.add_edges_from([(regulon_name, gene) for gene in target_genes])
plt.figure(figsize=(8, 8))
pos = nx.spring_layout(G)
nx.draw_networkx_nodes(G, pos, node_size=300, node_color="lightblue")
nx.draw_networkx_edges(G, pos, edge_color="gray", arrowstyle="->", arrowsize=10)
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title(f"Regulon: {regulon_name}")
plt.axis("off")
plt.show()



In [ ]:
target_genes